# 10 Performing Branch Analysis with Snapshots

Notebook generado a partir del paquete Python. Los módulos se incluyen en orden de dependencia (los módulos importados por otros aparecen primero).

> Nota: los `import` de módulos locales del paquete original se conservan tal cual. Como todos los módulos están combinados en este notebook en orden de dependencia, los símbolos referenciados ya quedan definidos en celdas anteriores.

## ¿Qué hace este notebook?

Realiza **análisis por ramas usando snapshots**. El flujo de revisión de código
(`set_prompt → build_llm_prompt → llm_propose_fix → finalize`, con `ChatAnthropic`)
se ejecuta una vez y queda registrado como checkpoints.

A partir de un **snapshot congelado** (vía `get_state_history` + `update_state`) se crean
varias ramas que cambian únicamente `review_style` (`concise`, `detailed`,
`performance-focused`), permitiendo **comparar los resultados lado a lado** sin re-ejecutar
todo desde cero.

> Requiere `ANTHROPIC_API_KEY` en el entorno (archivo `.env`).

## Ejemplo de uso

**Datos de interacción que espera el agente.** Tras la corrida base, el agente "continúa"
desde un **snapshot congelado**; el dato que inyectas para ramificar es `review_style`.

- Entrada inicial: `{"review_style": "concise"}`.
- Para **continuar/ramificar**: toma un `config` del historial (`get_state_history`),
  inyecta el nuevo valor con `update_state(snapshot_config, values={"review_style": ...})`
  y reanuda con `invoke(None, branch_config)`. Valores válidos de `review_style`:
  `"concise"`, `"detailed"`, `"performance-focused"`.

> Requiere `ANTHROPIC_API_KEY` en `.env`.

```python
import uuid

graph = build_graph()
base = {"configurable": {"thread_id": str(uuid.uuid4())}}
graph.invoke({"review_style": "concise"}, base)    # corrida base

history = list(graph.get_state_history(base))
snapshot_config = history[2].config                # punto del pasado a reutilizar

for style in ["concise", "detailed", "performance-focused"]:
    branch_config = graph.update_state(snapshot_config, values={"review_style": style})
    out = graph.invoke(None, branch_config)        # continúa la rama hasta concluir
    print(style, "->", out["output"][:80])
```

In [1]:
"""
You are building an AI agent that reviews buggy code and proposes fixes.
The workflow already runs correctly and produces a valid solution.
Now, you want to explore alternative review perspectives—concise, detailed, and performance-focused—without re-running everything
Using execution snapshots, you pause the workflow at a chosen checkpoint.
From the same frozen past, you branch into multiple futures.
Each branch changes only review_style, allowing precise, side-by-side comparison of outcomes.
"""

'\nYou are building an AI agent that reviews buggy code and proposes fixes.\nThe workflow already runs correctly and produces a valid solution.\nNow, you want to explore alternative review perspectives—concise, detailed, and performance-focused—without re-running everything\nUsing execution snapshots, you pause the workflow at a chosen checkpoint.\nFrom the same frozen past, you branch into multiple futures.\nEach branch changes only review_style, allowing precise, side-by-side comparison of outcomes.\n'

In [2]:

import os
import uuid
import hashlib
from typing_extensions import TypedDict, NotRequired
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_anthropic import ChatAnthropic

In [3]:


# --- State ---
class State(TypedDict):
    prompt: NotRequired[str]
    review_style: NotRequired[str]
    llm_prompt: NotRequired[str]
    fix: NotRequired[str]
    output: NotRequired[str]

In [4]:


# --- Nodes ---
def set_prompt(state: State):
    return {
        "prompt": (
            "Bug: `parse_port(env)` returns None when PORT is missing.\n"
            "Code:\n"
            "  def parse_port(env):\n"
            "      if 'PORT' in env:\n"
            "          return int(env['PORT'])\n"
            "      # missing return\n"
            "Expected: return 8080 if missing."
        )
    }

In [5]:

def build_llm_prompt(state: State):
    style = state.get("review_style", "concise")

    style_rules = {
        "concise": (
            "Output format:\n"
            "1) Only a Python code block.\n"
            "No explanation text."
        ),
        "detailed": (
            "Output format:\n"
            "1) Exactly 3 bullet points explaining the fix.\n"
            "2) Then a Python code block."
        ),
        "performance-focused": (
            "Output format:\n"
            "1) One sentence explaining performance/efficiency impact.\n"
            "2) Then a Python code block.\n"
            "Keep it tight."
        ),
    }

    rules = style_rules.get(style, style_rules["concise"])

    return {
        "llm_prompt": (
            "You are a code review assistant.\n"
            f"{state['prompt']}\n\n"
            f"Style: {style}\n"
            f"{rules}\n\n"
            "Must fix the missing return by providing a default port of 8080."
        )
    }

In [6]:

def llm_propose_fix(state: State):
    msg = model.invoke(state["llm_prompt"])
    return {"fix": msg.content}

In [7]:


def finalize(state: State):
    return {"output": f"Style: {state.get('review_style')}\n\n{state.get('fix')}"}

In [8]:


# --- Graph ---
def build_graph():
    g = StateGraph(State)
    g.add_node("set_prompt", set_prompt)
    g.add_node("build_llm_prompt", build_llm_prompt)
    g.add_node("llm_propose_fix", llm_propose_fix)
    g.add_node("finalize", finalize)

    g.add_edge(START, "set_prompt")
    g.add_edge("set_prompt", "build_llm_prompt")
    g.add_edge("build_llm_prompt", "llm_propose_fix")
    g.add_edge("llm_propose_fix", "finalize")
    g.add_edge("finalize", END)

    return g.compile(checkpointer=InMemorySaver())

In [9]:


# --- Helpers Functions ---
def show_checkpoints(graph, config):
    states = list(reversed(list(graph.get_state_history(config))))
    print("\n=== CHECKPOINTS ===")
    for i, s in enumerate(states):
        cid = s.config["configurable"]["checkpoint_id"]
        print(f"[{i}] next={s.next} | checkpoint_id={cid}")
    return states

In [10]:


def h10(text: str) -> str:
    return hashlib.sha256((text or "").encode("utf-8")).hexdigest()[:10]

In [11]:


def explain_snapshot(s):
    vals = dict(s.values or {})
    prompt = vals.get("prompt", "")
    llm_prompt = vals.get("llm_prompt", "")
    fix_exists = "fix" in vals

    print("\n=== SNAPSHOT YOU CHOSE ===")
    print(f"Resume will start at next node: {s.next}")
    print("What is identical across branches (the 'past'):")
    print(f"  prompt_hash     : {h10(prompt)}")
    print(f"  llm_prompt_hash : {h10(llm_prompt) if llm_prompt else '(not built yet)'}")
    print(f"  fix already made?: {fix_exists}")

    if s.next == ("llm_propose_fix",):
        print("\nInterpretation: We are paused RIGHT BEFORE the LLM decision.")
    elif s.next == ("finalize",):
        print("\nInterpretation: LLM already ran earlier; branching may not change the fix much.")
    elif s.next == ():
        print("\nInterpretation: This is the END checkpoint; nothing will run on resume.")
    else:
        print("\nInterpretation: You paused earlier; more of the pipeline will re-run on resume.")

In [12]:


def print_branch_results(results):
    print("\n=== BRANCH RESULTS (compare full fixes) ===")
    for r in results:
        print("\n" + "=" * 80)
        print(f"BRANCH {r['branch']} | review_style={r['review_style']}")
        print(f"Resumed from: {r['resume_next']}")
        print("-" * 80)
        print("Fix output:\n")
        print(r["fix"].strip() or "(no fix produced)")
    print("\n" + "=" * 80)

In [13]:


def main():
    load_dotenv()
    global model

    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError("ANTHROPIC_API_KEY not set.")

    model = ChatAnthropic(
        model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
        temperature=0,
    )

    graph = build_graph()

    thread_id = str(uuid.uuid4())
    base_config = {"configurable": {"thread_id": thread_id}}

    # 1) Baseline run to create checkpoint history
    print("\nRUN #1 (baseline) - creating checkpoint history")
    graph.invoke({"review_style": "concise"}, base_config)

    # 2) Choose checkpoint
    states = show_checkpoints(graph, base_config)
    idx = int(input("\nPick checkpoint index to branch from: ").strip())
    selected_state = states[idx]
    snapshot_config = selected_state.config

    explain_snapshot(selected_state)

    # 3) Branch from chosen snapshot
    branches = [
        ("A", "concise"),
        ("B", "detailed"),
        ("C", "performance-focused"),
    ]

    results = []
    for name, style in branches:
        branch_config = graph.update_state(snapshot_config, values={"review_style": style})
        out = graph.invoke(None, branch_config)
        results.append(
            {
                "branch": name,
                "review_style": style,
                "resume_next": str(selected_state.next),
                "fix": out.get("fix", out.get("output", "")),
            }
        )

    # 4) Show results in a compare-friendly way
    print_branch_results(results)

    print(
        "- All branches start from the SAME snapshot (same past).\n"
        "- Only `review_style` changes.\n"
        "- If you branched before `llm_propose_fix`, the LLM sees a different style in the prompt → different fix.\n"
    )

In [14]:


if __name__ == "__main__":
    main()


RUN #1 (baseline) - creating checkpoint history

=== CHECKPOINTS ===
[0] next=('__start__',) | checkpoint_id=1f157a12-36f0-65db-bfff-b3a6169cca8b
[1] next=('set_prompt',) | checkpoint_id=1f157a12-36f2-69ca-8000-f21ff85ed1be
[2] next=('build_llm_prompt',) | checkpoint_id=1f157a12-36f4-64f4-8001-af37b21ab17f
[3] next=('llm_propose_fix',) | checkpoint_id=1f157a12-36f5-6f42-8002-cdd032227ae5
[4] next=('finalize',) | checkpoint_id=1f157a12-4d74-67a4-8003-c193e81b6f25
[5] next=() | checkpoint_id=1f157a12-4d75-6c25-8004-80e02d50ad98

=== SNAPSHOT YOU CHOSE ===
Resume will start at next node: ('__start__',)
What is identical across branches (the 'past'):
  prompt_hash     : e3b0c44298
  llm_prompt_hash : (not built yet)
  fix already made?: False

Interpretation: You paused earlier; more of the pipeline will re-run on resume.

=== BRANCH RESULTS (compare full fixes) ===

BRANCH A | review_style=concise
Resumed from: ('__start__',)
--------------------------------------------------------------